In [26]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch import nn
from tqdm import tqdm

In [27]:
# PREDICTION VARIABLES
num_random_seeds = 10
num_awp_models = 1000
test_size = 0.20
validation_size = 0.25

NNEURON = 100
NLAYER = 1
NEPOCH = 250
BETA1 = 0.9
BETA2 = 0.999
TRAINLR = 0.001
SEARCHLR = 0.001
SEARCHMOMENTUM = 0.9
TARGET_VARIABLE = "qualified_gagne_3"

device = torch.device("mps" if torch.cuda.is_available() else "cpu")

fp_input = "../data/obermeyer/obermeyer_data_cleaned.csv"
fp_output = f"../data/predictions/awp_{TARGET_VARIABLE}.csv"

In [28]:
features = ['dem_female', 'dem_age_band_18-24_tm1', 'dem_age_band_25-34_tm1', 'dem_age_band_35-44_tm1', 'dem_age_band_45-54_tm1',
            'dem_age_band_55-64_tm1', 'dem_age_band_65-74_tm1', 'dem_age_band_75+_tm1', 'hypertension_elixhauser_tm1', 'cost_dialysis_tm1',
            'cost_emergency_tm1', 'cost_home_health_tm1', 'cost_ip_medical_tm1', 'cost_ip_surgical_tm1', 'cost_laboratory_tm1',
            'cost_op_primary_care_tm1', 'cost_op_specialists_tm1', 'cost_op_surgery_tm1', 'cost_other_tm1', 'cost_pharmacy_tm1',
            'cost_physical_therapy_tm1', 'cost_radiology_tm1', 'gagne_sum_tm1']
other_variables = ['person_id', 'gagne_sum_t', 'cost_t']

In [29]:
df = pd.read_csv(fp_input)
X = df[features + other_variables]
y = df[TARGET_VARIABLE]

In [30]:
# Multi-Layer Perceptron with weight perturbation
class MLP(nn.Module):
    def __init__(self, nn_arch):
        super().__init__()
        self.nfeature, self.nclass, self.nneuron, self.nlayer = nn_arch

        self.read_in = nn.Linear(self.nfeature, self.nneuron)
        self.ff = nn.Linear(self.nneuron, self.nneuron)
        self.read_out = nn.Linear(self.nneuron, self.nclass)

    def forward(self, x):
        x = self.read_in(x)
        for _ in range(self.nlayer):
            x = F.relu(self.ff(x))

        logits = self.read_out(x)
        return logits

def train_model(model, X, y, epoch, optimizer, criterion, device):
    for _ in range(epoch):
        model.train()
        optimizer.zero_grad()
        outputs = model(torch.Tensor(X).to(device))
        loss = criterion(torch.squeeze(outputs), torch.Tensor(y).type(torch.LongTensor).to(device))
        loss.backward()
        optimizer.step()

    return model

def evaluate(model, X, y, criterion, device):
    model.eval()
    logits = model(torch.Tensor(X).to(device))
    return criterion(torch.squeeze(logits), torch.Tensor(y).type(torch.LongTensor).to(device)).item()

def predict(model, X, class_target=1):
    model.eval()
    logits = model(torch.Tensor(X).to(device))
    pred = F.softmax(logits, dim=1)
    return pred[:, class_target].detach().cpu().numpy()

In [31]:
# Perturb the base model until the target instance has a prediction > likelihood tolerance, or the loss > loss tolerance
def get_awp_model(base_model, device, nn_arch, criterion, X_val, y_val, idx_target, class_target=1, pred_threshold=0.90, loss_tolerance=0.01):
    model = MLP(nn_arch).to(device)
    model.load_state_dict(copy.deepcopy(base_model.state_dict()))
    optimizer = torch.optim.SGD(model.parameters(), lr=SEARCHLR, momentum=SEARCHMOMENTUM)

    x_target = X_val[idx_target, :].reshape((1, X_val.shape[1]))
    target_logit = model(torch.Tensor(x_target).to(device))
    target_prob = torch.squeeze(F.softmax(target_logit, dim=1))
    baseline_loss = evaluate(model, X_val, y_val, criterion, device)
    val_loss = baseline_loss

    while target_prob[class_target]<pred_threshold and val_loss < baseline_loss + loss_tolerance:
        model.train()
        optimizer.zero_grad()
        target_logit = model(torch.Tensor(x_target).to(device))
        target_logit = torch.squeeze(target_logit)
        loss = -target_logit[class_target]
        loss.backward()
        optimizer.step()

        val_loss = evaluate(model, X_val, y_val, criterion, device)
        target_logit = model(torch.Tensor(x_target).to(device))
        target_prob = torch.squeeze(F.softmax(target_logit, dim=1))
    return model

In [32]:
output = []

for random_seed in range(num_random_seeds):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_seed)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=validation_size, random_state=random_seed)
    
    cost = X_test["cost_t"].to_numpy()
    gagne = X_test["gagne_sum_t"].to_numpy()
    person_id = X_test['person_id'].to_numpy()
    
    X_train = X_train.drop(columns=other_variables).to_numpy().copy()
    y_train = y_train.to_numpy().copy()
    X_val = X_val.drop(columns=other_variables).to_numpy().copy()
    y_val = y_val.to_numpy().copy()
    X_test = X_test.drop(columns=other_variables).to_numpy().copy()
    y_test = y_test.to_numpy().copy()

    np.random.seed(random_seed)
    random.seed(random_seed)
    torch.manual_seed(random_seed)
    nfeature = X_train.shape[1]
    nclass = len(set(y_train))
    nn_arch = [nfeature, nclass, NNEURON, NLAYER]

    base_model = MLP(nn_arch).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(base_model.parameters(), betas=[BETA1, BETA2], lr=TRAINLR)
    base_model = train_model(base_model, X_train, y_train, NEPOCH, optimizer, criterion, device)

    predictions = {}
    training_loss = {}
    validation_loss = {}

    for i in tqdm(range(num_awp_models)):
        model = get_awp_model(base_model, device, nn_arch, criterion, X_val, y_val, i)
        
        predictions[f'm_{i+1}'] = predict(model, X_test)
        training_loss[f'm_{i+1}'] = evaluate(model, X_train, y_train, criterion, device)
        validation_loss[f'm_{i+1}'] = evaluate(model, X_val, y_val, criterion, device)

    predictions_df = pd.concat([
        pd.DataFrame(predictions),
        pd.DataFrame(training_loss, index=[0]),
        pd.DataFrame(validation_loss, index=[0])]).reset_index(drop=True)
    
    predictions_df["y"] = np.concatenate([y_test, [np.nan, np.nan]])
    predictions_df["person_id"] = np.concatenate([person_id, [-2, -1]]) # -1 indicates validation loss, -2 indicates training loss
    predictions_df['cost_t'] = np.concatenate([cost, [np.nan, np.nan]]) 
    predictions_df['gagne_sum_t'] = np.concatenate([gagne, [np.nan, np.nan]])
    predictions_df["seed"] = random_seed

    output.append(predictions_df)
    
output = pd.concat(output)
for c in output.columns:
    if c.startswith("m_"):
        output[c] = output[c].astype('float32')

100%|██████████| 1000/1000 [00:11<00:00, 89.79it/s]


In [33]:
output.tail()

,m_1,m_2,m_3,m_4,m_5,m_6,m_7,m_8,m_9,m_10,...,m_996,m_997,m_998,m_999,m_1000,y,person_id,cost_t,gagne_sum_t,seed
9754,0.714173,0.717564,0.722787,0.721496,0.720533,0.716291,0.723093,0.717690,0.719742,0.720666,...,0.722129,0.723550,0.717419,0.674851,0.713648,1.0,28651,3500.0,3.0,9
9755,0.009499,0.009880,0.010192,0.010084,0.007721,0.007766,0.010120,0.009908,0.008909,0.007804,...,0.007952,0.010144,0.009869,0.006308,0.009459,0.0,21611,0.0,0.0,9
9756,0.963281,0.963762,0.964719,0.964474,0.966425,0.963627,0.965023,0.963790,0.965657,0.966368,...,0.964585,0.965057,0.963737,0.959842,0.963192,1.0,965,46400.0,8.0,9
9757,0.240465,0.232616,0.235428,0.247100,0.233124,0.232120,0.249347,0.240407,0.238585,0.233598,...,0.234482,0.234943,0.232560,0.223841,0.235871,NaN,-2,NaN,NaN,9
9758,0.232544,0.223278,0.226468,0.239662,0.224927,0.223287,0.242149,0.232839,0.230474,0.225478,...,0.225899,0.225810,0.223216,0.212586,0.227907,NaN,-1,NaN,NaN,9


In [34]:
output.to_csv(fp_output, index=False)